# Subm3 - g8 MEI A

Submissão `A` para a Subm3 com um `Soft Voting` entre os 3 melhores modelos.

(Pesos do DistilBERT tiveram de ficar fora do GitHub pois o mesmo não suporta ficheiros daquele tamanho)


In [ ]:
import os
import sys
import pickle
import json

sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.data_processing import clean_text, clean_text_sequences
from src.features import TextDataset, Vocabulary, load_glove_embeddings, texts_to_sequences
from src.models_numpy.dnn import CategoricalCrossEntropy, Dataset, NeuralNetwork, accuracy
from src.models_pytorch.cnn1d import CNN1DClassifier
from src.models_pytorch.distilbert import DistilBERTClassifier, DistilBERTDataset, get_tokenizer


In [ ]:
ROOT = os.path.abspath('..')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']

SUBM3_INPUT_PATH = os.path.join(ROOT, 'Subm3', 'validation_data', 'subm3.csv')
SUBM3_OUTPUT_PATH = os.path.join(ROOT, 'Subm3', 'subm3-g8-MEI-A.csv')
SAVED_MODELS_DIR = os.path.join(ROOT, 'saved_models')
GLOVE_PATH = os.path.join(ROOT, 'data', 'embeddings', 'glove.6B.100d.txt')

print(f'Device: {DEVICE}')

def soft_voting(probabilities, weights=None):
    if weights is None:
        weights = [1.0 / len(probabilities)] * len(probabilities)
    return sum(weight * probs for weight, probs in zip(weights, probabilities))

## Load Subm3


In [ ]:
df_subm3 = pd.read_csv(SUBM3_INPUT_PATH, sep=';')
print(df_subm3.head())
print(f'Total rows: {len(df_subm3)}')


## Load DNN


In [ ]:
dnn_model = NeuralNetwork(loss=CategoricalCrossEntropy, metric=accuracy, verbose=False)
dnn_model.load(os.path.join(SAVED_MODELS_DIR, 'dnn_final_model.npz'))

with open(os.path.join(SAVED_MODELS_DIR, 'dnn_final_vectorizer.pkl'), 'rb') as f:
    dnn_vectorizer = pickle.load(f)


def predict_dnn(model, vectorizer, texts_raw):
    texts_clean = [clean_text(text) for text in texts_raw]
    X = vectorizer.transform(texts_clean, texts_raw)
    return model.predict(Dataset(X, np.zeros((len(texts_raw), len(CLASSES)))))


## Load CNN1D


In [ ]:
with open(os.path.join(SAVED_MODELS_DIR, 'cnn1d_final_config.json'), 'r') as f:
    cnn_cfg = json.load(f)

cnn_vocab = Vocabulary(max_words=cnn_cfg['vocab_size'])
cnn_vocab.load(os.path.join(SAVED_MODELS_DIR, 'cnn1d_final_vocab.pkl'))

style_stats = np.load(os.path.join(SAVED_MODELS_DIR, 'cnn1d_final_style_stats.npz'))
style_mean = style_stats['mean']
style_std = style_stats['std']

with open(os.path.join(SAVED_MODELS_DIR, 'cnn1d_final_style_extractor.pkl'), 'rb') as f:
    cnn_style_extractor = pickle.load(f)

cnn_embeddings = None
if os.path.exists(GLOVE_PATH):
    cnn_embeddings = load_glove_embeddings(GLOVE_PATH, cnn_vocab, embedding_dim=cnn_cfg['embedding_dim'])

cnn_model = CNN1DClassifier(
    vocab_size=len(cnn_vocab),
    embedding_dim=cnn_cfg['embedding_dim'],
    n_filters=cnn_cfg['n_filters'],
    filter_sizes=cnn_cfg['filter_sizes'],
    output_dim=len(CLASSES),
    dropout=cnn_cfg['dropout'],
    pretrained_embeddings=cnn_embeddings,
    n_style_features=len(style_mean),
).to(DEVICE)

cnn_model.load_state_dict(torch.load(os.path.join(SAVED_MODELS_DIR, 'cnn1d_final_model.pt'), map_location=DEVICE))
cnn_model.eval()


def predict_cnn(model, vocab, style_extractor, style_mean, style_std, texts_raw):
    texts_clean = [clean_text_sequences(text) for text in texts_raw]
    sequences = texts_to_sequences(texts_clean, vocab, max_len=cnn_cfg['max_len'])
    style = style_extractor.transform(texts_raw)
    style = (style - style_mean) / style_std
    dataset = TextDataset(sequences, np.zeros(len(texts_raw), dtype=np.int64), style)
    loader = DataLoader(dataset, batch_size=cnn_cfg['batch_size'])
    probs = []
    with torch.no_grad():
        for seqs, style_batch, _ in loader:
            seqs = seqs.to(DEVICE)
            style_batch = style_batch.to(DEVICE)
            probs.append(torch.softmax(model(seqs, style_batch), dim=1).cpu().numpy())
    return np.vstack(probs)


## Load DistilBERT


In [ ]:
bert_cfg = {'dropout': 0.1, 'batch_size': 8, 'max_len': 128}
bert_tokenizer = get_tokenizer()
bert_model = DistilBERTClassifier(output_dim=len(CLASSES), dropout=bert_cfg['dropout'], freeze_bert=False).to(DEVICE)
bert_model.load_state_dict(torch.load(os.path.join(SAVED_MODELS_DIR, 'distilbert_final_model.pt'), map_location=DEVICE))
bert_model.eval()


def predict_bert(model, texts):
    labels = np.zeros(len(texts), dtype=np.int64)
    dataset = DistilBERTDataset(texts, labels, bert_tokenizer, max_len=bert_cfg['max_len'])
    loader = DataLoader(dataset, batch_size=bert_cfg['batch_size'])
    probs = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            logits = model(input_ids, attention_mask)
            probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(probs)


## Predict and Save CSV


In [ ]:
texts_subm3 = df_subm3['Text'].tolist()

dnn_probs = predict_dnn(dnn_model, dnn_vectorizer, texts_subm3)
cnn_probs = predict_cnn(cnn_model, cnn_vocab, cnn_style_extractor, style_mean, style_std, texts_subm3)
bert_probs = predict_bert(bert_model, texts_subm3)

ensemble_probs = soft_voting([dnn_probs, cnn_probs, bert_probs])
pred_ids = np.argmax(ensemble_probs, axis=1)
pred_labels = [CLASSES[idx] for idx in pred_ids]

submission = pd.DataFrame({
    'ID': df_subm3['ID'],
    'Label': pred_labels,
})
submission.to_csv(SUBM3_OUTPUT_PATH, sep=';', index=False)

print(submission.head())
print(f'Saved to {SUBM3_OUTPUT_PATH}')
